# Demo: Bessel-Sobolev Operators on a 3D Box Domain

This notebook demonstrates the `BesselSobolevBox` spectral operator family in `pygeoinf3d`.

Given orthonormal eigenpairs $(\Phi_k, \lambda_k)$ of the negative Laplacian $-\Delta$ on a box domain $\Omega \subset \mathbb{R}^3$, the Bessel potential operator of order $s \in \mathbb{R}$ is defined spectrally as:

$$
(\mathcal{B}^s f)(\mathbf{x}) = \sum_{k=0}^{N-1} (\tau + \lambda_k)^{s/2} \langle f, \Phi_k \rangle \Phi_k(\mathbf{x})
$$

where $\tau > 0$ is a shift parameter.

This operator plays two key roles:
1.  **Metric Operator ($s > 0$):** Defines the Sobolev norm $\|f\|_{H^s} = \|\mathcal{B}^s f\|_{L^2}$.
2.  **Covariance Operator ($s < 0$):** Defines the covariance $C = \mathcal{B}^{2s} = (\tau - \Delta)^{s}$ for Gaussian random fields with Matérn-like correlations.

**Visualisation mode:** In the visualisation sections below you can choose `"static"`, `"interactive"`, or `"both"` and re-run the cells.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pygeoinf3d import BoxDomain, Lebesgue3D, TensorProductProvider
from pygeoinf3d import BesselSobolevBox, KLSampler3D
from pygeoinf3d.core.functions import Function3D
from pygeoinf3d.plot import plot_function3d_axis_slice

print("Imports successful.")

## Section 1: Visualisation Mode Configuration

Configure the plotting style. `"interactive"` uses Plotly-based sliders; `"static"` uses Matplotlib embedded plots.

In [ ]:
VISUALIZATION_MODE = "interactive"  # Options: "static", "interactive", "both"
INTERACTIVE_AXIS = "z"
INTERACTIVE_N_SLICES = 31
INTERACTIVE_RESOLUTION = 70

try:
    from pygeoinf3d.plot import plot_function3d_axis_slider
    INTERACTIVE_PLOTTING_AVAILABLE = True
except ImportError:
    plot_function3d_axis_slider = None
    INTERACTIVE_PLOTTING_AVAILABLE = False

valid_modes = {"static", "interactive", "both"}
if VISUALIZATION_MODE not in valid_modes:
    raise ValueError(f"VISUALIZATION_MODE must be one of {sorted(valid_modes)}, got {VISUALIZATION_MODE!r}")

def _show_interactive_functions(functions, titles):
    if not INTERACTIVE_PLOTTING_AVAILABLE:
        print("Interactive plotting requested, but Plotly is not installed.")
        return

    if len(functions) != len(titles):
        raise ValueError("functions and titles must have the same length.")

    for fn, title in zip(functions, titles):
        fig = plot_function3d_axis_slider(
            fn,
            axis=INTERACTIVE_AXIS,
            n_slices=INTERACTIVE_N_SLICES,
            resolution=INTERACTIVE_RESOLUTION,
            show_plot=False,
            title=title,
        )
        fig.show()

print(f"VISUALIZATION_MODE = {VISUALIZATION_MODE!r}")
print(f"Interactive plotting available: {INTERACTIVE_PLOTTING_AVAILABLE}")

## Section 2: Domain and Spectral Basis Setup

We use the unit cube $\Omega = [0, 1]^3$ with **Neumann boundary conditions**.
Neumann BCs imply the eigenfunctions are products of cosines:

$$
\Phi_{\mathbf{k}}(\mathbf{x}) = \prod_{i=1}^3 \phi_{k_i}(x_i), \quad \phi_0(t) = 1, \quad \phi_k(t) = \sqrt{2} \cos(k \pi t)
$$

The zero mode (constant function) corresponds to $\lambda_0 = 0$. The shift parameter $\tau > 0$ is essential here to ensure invertibility, as $(\tau + 0)^{s/2}$ is well-defined and non-zero.

In [ ]:
# Domain: unit cube [0,1]^3 with Neumann BCs
box   = BoxDomain(np.zeros(3), np.ones(3))
space = Lebesgue3D(box)

# Spectral provider: 20×20×20 = 8000 Neumann modes
TRUNCATION = (20, 20, 20)
provider = TensorProductProvider(box, truncation=TRUNCATION, bc_type='neumann')

print(f"Provider dimension: {provider.dim} modes")
print(f"Boundary conditions: {provider.bc_type}")
print(f"Smallest Laplacian eigenvalue: {provider.get_eigenvalue(0):.4f}"
      "  (zero mode — Neumann zero-frequency constant)")
print(f"Largest  Laplacian eigenvalue: {provider.get_eigenvalue(provider.dim - 1):.4f}")


## Section 3: Positive Orders (Smoothing / Metric Operators)

When $s > 0$, the operator $\mathcal{B}^s = (\tau - \Delta)^{s/2}$ amplifies high-frequency modes (large $\lambda_k$), acting like a derivative operator.

Let's visualize the eigenvalue growth for different orders.

In [ ]:
SHIFT = 0.1   # strictly positive: keeps (shift + lambda_k) > 0 for all modes

# Sort modes by base Laplacian eigenvalue for a consistent x-axis
base_eigs = np.array([provider.get_eigenvalue(k) for k in range(provider.dim)])
sort_idx  = np.argsort(base_eigs)

# --- Positive orders: eigenvalue growth ---
fig, ax = plt.subplots(figsize=(8, 5))
for s in [0.0, 0.5, 1.0, 2.0]:
    op = BesselSobolevBox(space, provider, shift=SHIFT, order=s)
    mu = np.array([op.get_eigenvalue(k) for k in range(provider.dim)])
    ax.semilogy(np.arange(provider.dim), mu[sort_idx], lw=1.5, label=f's = {s}')

ax.set_xlabel("Mode rank (sorted by Laplacian eigenvalue)")
ax.set_ylabel(r"$\mu_k = (\tau + \lambda_k)^{s/2}$")
ax.set_title(rf"Eigenvalue spectrum — positive orders  ($\tau = {SHIFT}$)")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()


### Negative Orders — Eigenvalue Decay

For $s < 0$, the eigenvalues $\mu_k = (\tau + \lambda_k)^{s/2}$ **decay** with mode index.
This is the key property that makes $\mathcal{B}^s$ (with $s < 0$) a valid Gaussian covariance operator:
it is bounded, self-adjoint, positive, and — for $s < -3/2$ in 3D — trace-class (sum of eigenvalues finite).


In [ ]:
# --- Negative orders: eigenvalue decay ---
fig, ax = plt.subplots(figsize=(8, 5))
for s in [-0.5, -1.0, -2.0]:
    op = BesselSobolevBox(space, provider, shift=SHIFT, order=s)
    mu = np.array([op.get_eigenvalue(k) for k in range(provider.dim)])
    ax.semilogy(np.arange(provider.dim), mu[sort_idx], lw=1.5, label=f's = {s}')

ax.set_xlabel("Mode rank (sorted by Laplacian eigenvalue)")
ax.set_ylabel(r"$\mu_k = (\tau + \lambda_k)^{s/2}$")
ax.set_title(rf"Eigenvalue spectrum — negative orders  ($\tau = {SHIFT}$)")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()


### Action on a Test Function

We define a smooth test function (Gaussian bump) and apply operators of increasing order. 
Higher positive orders correspond to higher derivatives, which should amplify variations.

In [ ]:
# Smooth Gaussian bump centred in the unit cube
def gaussian_bump(x):
    x      = np.asarray(x, dtype=float)
    centre = np.array([0.5, 0.5, 0.5])
    r2     = np.sum((x - centre) ** 2, axis=-1)
    return np.exp(-20.0 * r2)

f_input = Function3D(box, evaluate_callable=gaussian_bump)

# Apply for orders 0 (identity), 1, 2 — just call op(f) directly
orders_pos = [0.0, 1.0, 2.0]
funcs_pos  = []
titles_pos = []

for s in orders_pos:
    op = BesselSobolevBox(space, provider, shift=SHIFT, order=s)
    funcs_pos.append(op(f_input))
    titles_pos.append(f"s = {s}" + ("  (identity)" if s == 0.0 else ""))

if VISUALIZATION_MODE in {"static", "both"}:
    fig, axes = plt.subplots(1, len(orders_pos), figsize=(13, 4))
    for ax, fn, title in zip(axes, funcs_pos, titles_pos):
        plot_function3d_axis_slice(
            fn, axis='z', coord=0.5, resolution=50,
            ax=ax, show_plot=False, title=title,
        )
    plt.suptitle(
        r"Gaussian bump under $\mathcal{B}^s$  (z = 0.5 slice)", fontsize=12
    )
    plt.tight_layout()
    plt.show()

if VISUALIZATION_MODE in {"interactive", "both"}:
    _show_interactive_functions(funcs_pos, titles_pos)


## Section 4: Negative Orders — Sampling from a Gaussian Random Field

When $s < 0$, the operator $\mathcal{B}^s = (\tau - \Delta)^{s/2}$ is a valid covariance
operator for a Gaussian measure $\mathcal{N}(0,\, \mathcal{B}^s)$.
We draw samples using `KLSampler3D` via the truncated KL expansion:

$$
u(\mathbf{x}) = \sum_{k=0}^{K-1} \sqrt{\mu_k}\, \xi_k\, \Phi_k(\mathbf{x}),
\qquad \xi_k \sim \mathcal{N}(0, 1), \quad \mu_k = (\tau + \lambda_k)^{s/2}.
$$

**Smoothness rule:** more negative $s$ → faster eigenvalue decay → smoother samples.
The positive shift $\tau$ prevents any singularity at the Neumann zero mode $\lambda_0 = 0$.


In [ ]:
N_KL_MODES      = provider.dim          # use all available modes
sampling_orders = [-1.0, -2.0, -4.0]

sample_fns    = []
sample_titles = []

for s in sampling_orders:
    cov_op  = BesselSobolevBox(space, provider, shift=SHIFT, order=s)
    sampler = KLSampler3D(cov_op, n_modes=N_KL_MODES, rng=np.random.default_rng(42))
    sample_fns.append(sampler.sample())
    sample_titles.append(f"Sample  s = {s}")

print(f"Generated {len(sample_fns)} samples  (K = {N_KL_MODES} KL modes each).")


### Visualizing Samples

Observe how the samples become smoother (less high-frequency noise) as the order becomes more negative.
- `order=-1.0`: Very rough (barely regularized)
- `order=-2.0`: "Green's function" smoothness (like $(I-\Delta)^{-1}$)
- `order=-4.0`: Very smooth

In [ ]:
if VISUALIZATION_MODE in {"static", "both"}:
    fig, axes = plt.subplots(1, len(sampling_orders), figsize=(13, 4))
    for ax, fn, title in zip(axes, sample_fns, sample_titles):
        plot_function3d_axis_slice(
            fn, axis='z', coord=0.5, resolution=60,
            ax=ax, show_plot=False, title=title, cmap="RdBu_r",
        )
    plt.suptitle(
        r"KL samples from $\mathcal{N}(0,\,(\tau - \Delta)^{s})$"
        "  (z = 0.5 slice)",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()

if VISUALIZATION_MODE in {"interactive", "both"}:
    _show_interactive_functions(sample_fns, sample_titles)


## Summary

| `order` (`s`) | Role | Effect on modes |
|---|---|---|
| `s > 0` | Metric / pseudo-differential | Amplifies high frequencies — acts like fractional derivative |
| `s = 0` | Identity | No change |
| `s < 0` | Covariance / smoothing | Attenuates high frequencies — Matérn-like GP covariance |

| Parameter | Larger value | Smaller value |
|---|---|---|
| `shift` (τ) | Compresses ratio between low/high-freq eigenvalues | Sharpens the contrast |
| `\|order\|` (s < 0) | Smoother samples (faster spectral decay) | Rougher samples |
| `n_modes` in KL | More fine-scale variability in samples | Faster sampling |

**Key API:**
```python
op      = BesselSobolevBox(space, provider, shift=τ, order=s)
result  = op(f)                                        # Function3D → Function3D

sampler = KLSampler3D(op, n_modes=K, rng=rng)         # GP prior from covariance op
sample  = sampler.sample()                             # one Function3D draw
samples = sampler.samples(n)                           # list of n draws
```

**Next step:** wrap `BesselSobolevBox` in `pygeoinf.OperatorWeightedHilbertSpace` to obtain
a fully functional $H^s$ Sobolev space equipped with the Bessel-Sobolev inner product
$\langle f, g \rangle_{H^s} = \langle \mathcal{B}^s f,\, \mathcal{B}^s g \rangle_{L^2}$.

See also [`kl_sampler_prior_demo.ipynb`](kl_sampler_prior_demo.ipynb) for a deeper treatment
of KL sampling as a Bayesian prior, and [`6_laplacian_3d_demo.ipynb`](6_laplacian_3d_demo.ipynb)
for comparison with the `GeneralizedBoxLaplacian` family.
